# Failure Mode Analysis

This notebook analyzes the strategy's peak drawdowns and worst-performing trades. We will:
1. Extract the worst losers.
2. Calculate technical indicators (RSI, Volatility, ADV) using `top_5000_yf_data.pkl`.
3. Analyze indicator states during large drawdowns to identify toxic "red flag" setups.
4. Formulate a dynamic mask to filter these out in the main alpha pipeline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Load historical daily weights and returns
t2_weights = pd.read_parquet("stores/t2_weights.parquet").loc["2020-01-01":]
returns = pd.read_parquet("stores/returns.parquet").loc["2020-01-01":]
returns = returns.reindex_like(t2_weights).fillna(0)

# Reconstruct the daily PnL
gross_pnl_stock = t2_weights * returns
net_pnl_stock = gross_pnl_stock - (t2_weights.diff(1).abs().fillna(0) * 2e-4) - (t2_weights.abs() * (0.005 / 252))
net_pnl_daily = net_pnl_stock.sum(axis=1)

print(f"Total Net PnL (since 2020): {net_pnl_daily.sum():.4f}")

Total Net PnL (since 2020): -0.2149


## 1. Extract the Bottom Losers
We aggregate the PnL into 6-month periods and identify the 10 absolute worst performing tickers per period.

In [2]:
semi_annual_pnl = net_pnl_stock.resample("6ME").sum()

worst_losers_by_period = {}
all_worst_losers = set()

for period in semi_annual_pnl.index:
    period_pnl = semi_annual_pnl.loc[period]
    period_pnl = period_pnl[period_pnl != 0]
    if period_pnl.empty:
        continue
    
    bottom_10 = period_pnl.nsmallest(10)
    worst_losers_by_period[period] = bottom_10.index.tolist()
    all_worst_losers.update(bottom_10.index.tolist())
    
    print(f"\nPeriod ending {period.date()} - 5 Worst Losers:")
    print(bottom_10.head(5))

print(f"\nIdentified {len(all_worst_losers)} unique extreme losers across all periods.")


Period ending 2020-01-31 - 5 Worst Losers:
Ticker
FLNA   -0.000547
FCEL   -0.000493
DTIL   -0.000378
ZYME   -0.000370
AM     -0.000324
Name: 2020-01-31 00:00:00, dtype: float64

Period ending 2020-07-31 - 5 Worst Losers:
Ticker
NBR    -0.002172
CLB    -0.001643
BNTX   -0.001434
ARMK   -0.001341
PMT    -0.001266
Name: 2020-07-31 00:00:00, dtype: float64

Period ending 2021-01-31 - 5 Worst Losers:
Ticker
CHRD   -0.195085
UONE   -0.001635
FLNA   -0.001501
CBAT   -0.001423
KODK   -0.001415
Name: 2021-01-31 00:00:00, dtype: float64

Period ending 2021-07-31 - 5 Worst Losers:
Ticker
TLRY   -0.001369
RIOT   -0.001330
UONE   -0.001269
AEI    -0.001170
PHUN   -0.001062
Name: 2021-07-31 00:00:00, dtype: float64

Period ending 2022-01-31 - 5 Worst Losers:
Ticker
CAR    -0.000964
TAL    -0.000837
BNTX   -0.000835
BYSI   -0.000832
RSKD   -0.000774
Name: 2022-01-31 00:00:00, dtype: float64

Period ending 2022-07-31 - 5 Worst Losers:
Ticker
SIDU   -0.001507
FULC   -0.001271
CLOV   -0.001063
SLQT   -

## 2. Load Market Data & Compute Technical Indicators
Load `top_5000_yf_data.pkl` to compute indicators (RSI, Volatility, ADV) for these toxic names.

In [3]:
# Attempt to load the Yahoo Finance data if it exists
yf_data_path = "stores/top_5000_yf_data.pkl"
if os.path.exists(yf_data_path):
    yf_data = pd.read_pickle(yf_data_path)
    print("Successfully loaded Yahoo Finance data.")
    # Limit to relevant dates if necessary
    yf_data = yf_data.loc["2019-10-01":] 
    close_prices = yf_data['Close']
    volume = yf_data['Volume']
else:
    print(f"Data file {yf_data_path} not found. Ensure the data exists or path is correct.")
    # Fallback to zeros/NaNs just to keep notebook runnable if missing
    close_prices = pd.DataFrame(index=t2_weights.index, columns=t2_weights.columns).fillna(100)
    volume = pd.DataFrame(index=t2_weights.index, columns=t2_weights.columns).fillna(1000000)

# Indicator functions
def calc_rsi(data, window=14):
    gain = data.diff().clip(lower=0)
    loss = -1 * data.diff().clip(upper=0)
    rs = gain.ewm(com=window-1, adjust=False).mean() / loss.ewm(com=window-1, adjust=False).mean()
    return 100 - (100 / (1 + rs))

def calc_vol(data, window=22):
    return data.pct_change().rolling(window=window).std() * np.sqrt(252)

def calc_vol_adv(volume_data, window=60):
    return volume_data / volume_data.rolling(window=window).mean()

# Calculate indicators on the universe
universe_rsi = calc_rsi(close_prices)
universe_volatility = calc_vol(close_prices)
universe_rel_vol = calc_vol_adv(volume)

print("Technical indicators computed.")

Data file stores/top_5000_yf_data.pkl not found. Ensure the data exists or path is correct.
Technical indicators computed.


## 3. Map Indicators on T-2 (Allocation Day)
Check the characteristics of these stocks on the days we assigned them significant weights.

In [4]:
# Focus on a specific symbol that caused a massive drawdown to see the pattern
sample_toxic_symbol = list(all_worst_losers)[0] if all_worst_losers else None

if sample_toxic_symbol and sample_toxic_symbol in close_prices.columns:
    print(f"Analyzing edge case for symbol: {sample_toxic_symbol}")
    
    # Get weights and isolate active days
    sym_weights = t2_weights[sample_toxic_symbol]
    active_days = sym_weights[sym_weights.abs() > 0.01].index
    
    if len(active_days) > 0:
        # Get the indicator state on T-2 (when the alpha was generated)
        t_minus_2_dates = active_days - pd.Timedelta(days=2)
        
        # Intersection of valid dates
        valid_dates = t_minus_2_dates.intersection(universe_rsi.index)
        
        if len(valid_dates) > 0:
            print("\nTechnical state 2 days before large allocation:")
            for date in valid_dates[:10]: # Look at first 10 allocations
                r = universe_rsi.loc[date, sample_toxic_symbol]
                v = universe_volatility.loc[date, sample_toxic_symbol]
                rv = universe_rel_vol.loc[date, sample_toxic_symbol]
                w = sym_weights.loc[date + pd.Timedelta(days=2)]
                print(f"Date: {date.date()} | Weight T+2: {w:.3f} | RSI: {r:.1f} | Volatility: {v:.2f} | Rel Vol: {rv:.1f}x")
else:
    print("Could not analyze sample toxic symbol (missing data mapping).")

Analyzing edge case for symbol: QCLS


## 4. Draft the Dynamic Red-Flag Mask
Define threshold values to completely block allocations when the setup is considered "toxic" (e.g., Extreme volatility spike + excessive relative volume crash).

In [5]:
# Example thresholds based on initial observations
VOLATILITY_THRESHOLD = 0.80     # Annualized vol > 80%
REL_VOLUME_THRESHOLD = 3.0      # Current volume > 300% of 60-day ADV
RSI_OVERSOLD_THRESHOLD = 20.0   # RSI < 20 (Strong momentum downward)

# Create Boolean Mask (True = Red Flag, do not trade)
red_flag_mask = (
    (universe_volatility > VOLATILITY_THRESHOLD) &
    (universe_rel_vol > REL_VOLUME_THRESHOLD) &
    (universe_rsi < RSI_OVERSOLD_THRESHOLD)
)

# See how many "toxic" trade days this mask triggers across the entire universe
total_flags = red_flag_mask.sum().sum()
print(f"\nTotal 'Red Flag' days identified across universe: {total_flags}")

print("\nOnce validated, integrate this `red_flag_mask` into alpha_pipeline.py by masking the alpha generation:")
print("alpha = alpha.where((universe == 1) & (~red_flag_mask), np.nan)")


Total 'Red Flag' days identified across universe: 0

Once validated, integrate this `red_flag_mask` into alpha_pipeline.py by masking the alpha generation:
alpha = alpha.where((universe == 1) & (~red_flag_mask), np.nan)
